In [8]:
# ================================================================
# TORCH: Factory-Level Risk Index — Myanmar Apparel Supply Chains
# Stage 1: Data Loading & Integration
# ================================================================

import pandas as pd
import re
import requests
import time
from bs4 import BeautifulSoup
from thefuzz import process  # pip install thefuzz python-Levenshtein

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/122.0.0.0 Safari/537.36"}

# ----------------------------------------------------------------
# 1. Load the three datasets
# ----------------------------------------------------------------

news  = pd.read_csv("myanmarlabournews_full.csv")
osh   = pd.read_csv("facilities.csv")
bhrrc = pd.read_csv("myanmargarmentworkerallegationsdatabase.csv")

# ----------------------------------------------------------------
# 2. Clean Myanmar Labour News
# ----------------------------------------------------------------

news = news[["Title", "Tags", "URL", "Content"]].copy()

# The date is embedded in the Title, e.g. "...Some Dismissed Mar 17, 2026"
def extract_date(title):
    match = re.search(r"(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+\d{1,2},\s+\d{4}", str(title))
    return match.group(0) if match else None

news["date"] = news["Title"].apply(extract_date)
news["date"] = pd.to_datetime(news["date"], errors="coerce")

# Remove the date that got appended to the title
news["title"] = news["Title"].apply(lambda t: re.sub(
    r"(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+\d{1,2},\s+\d{4}", "", str(t)
).strip())

news = news.drop(columns=["Title"])

# ----------------------------------------------------------------
# 3. Re-scrape content using the simple paragraph method
#    (the original scraped content had encoding issues and
#     included non-body text like author names)
# ----------------------------------------------------------------

def scrape_content(url):
    try:
        res = requests.get(url, headers=HEADERS, timeout=20)
        soup = BeautifulSoup(res.text, "html.parser")
        body = " ".join([p.get_text() for p in soup.find_all("p") if len(p.get_text()) > 50])
        return body
    except Exception as e:
        print(f"Failed: {url} — {e}")
        return None

print("Re-scraping article content...")
for i, row in news.iterrows():
    print(f"[{i+1}/{len(news)}] {str(row['title'])[:60]}")
    content = scrape_content(row["URL"])
    if content:
        news.at[i, "Content"] = content
    time.sleep(1.5)

print("Myanmar Labour News loaded:", len(news), "articles")
print("Date range:", news["date"].min(), "to", news["date"].max())

# ----------------------------------------------------------------
# 4. Clean OSH Facilities
# ----------------------------------------------------------------

osh = osh[[
    "os_id", "name", "address", "lat", "lng",
    "number_of_workers", "parent_company",
    "facility_type", "product_type", "is_closed"
]].copy()

osh["name_lower"] = osh["name"].str.lower().str.strip()

print("\nOSH Facilities loaded:", len(osh), "factories")

# ----------------------------------------------------------------
# 5. Clean BHRRC Allegations
# ----------------------------------------------------------------

bhrrc.columns = bhrrc.columns.str.strip()

bhrrc = bhrrc.rename(columns={
    "Factory"           : "factory_name",
    "Date Reported"     : "date_reported",
    "Number Affected"   : "num_affected",
    "Associated Buyers" : "buyers",
    "Allegation Summary": "allegation_summary",
    "Issue Groups"      : "issue_groups",
    "Response Action"   : "response_action",
    "Source Link"       : "source_link"
})

bhrrc["date_reported"] = pd.to_datetime(bhrrc["date_reported"], errors="coerce")
bhrrc["factory_lower"] = bhrrc["factory_name"].str.lower().str.strip()

print("\nBHRRC Allegations loaded:", len(bhrrc), "records")
print("Unique factories:", bhrrc["factory_name"].nunique())

# ----------------------------------------------------------------
# 6. Map BHRRC issue groups to ILO/Better Work CAT clusters
#
# Source: Better Work Global Compliance Assessment Tool (ILO & IFC, 2025)
# https://betterwork.org/wp-content/uploads/Better-Work-Global-Compliance-Assessment-Tool.pdf
#
# The Better Work CAT organises labour compliance into 8 clusters,
# each grounded in ILO core conventions. BHRRC issue group labels
# are mapped to the corresponding CAT cluster below.
# ----------------------------------------------------------------

CAT_MAPPING = {
    "child_labour"          : ["Child labour"],                              # C138, C182
    "forced_labour"         : ["Harassment intimidation and abuse"],         # C29, C105
    "discrimination"        : ["Gender-based violence and harassment"],      # C100, C111
    "freedom_of_association": ["Attacks on freedom of association"],         # C87, C98
    "working_hours"         : ["Inhumane work rates and mandatory overtime",
                               "Denial of leave"],                           # C1
    "compensation"          : ["Reduced wages and wage theft"],              # C95, C131
    "osh"                   : ["Unsafe working conditions"],                 # C155
    "contracts"             : ["Denial of permanent contracts"],             # C158
}

# ----------------------------------------------------------------
# 7. Aggregate BHRRC to one row per factory with binary labels
# ----------------------------------------------------------------

def collect_issues(series):
    issues = []
    for val in series.dropna():
        issues.extend([i.strip() for i in val.split(",")])
    return list(set(issues))

bhrrc_by_factory = (
    bhrrc.groupby("factory_name")
    .agg(
        allegation_count = ("factory_name", "count"),
        latest_date      = ("date_reported", "max"),
        total_affected   = ("num_affected", "sum"),
        all_issues       = ("issue_groups", collect_issues),
        buyers           = ("buyers", lambda x: "; ".join(x.dropna().unique())),
    )
    .reset_index()
)

for cluster, keywords in CAT_MAPPING.items():
    bhrrc_by_factory[f"label_{cluster}"] = bhrrc_by_factory["all_issues"].apply(
        lambda issues: int(any(k in issues for k in keywords))
    )

print("\nFactory-level label table built:", len(bhrrc_by_factory), "factories")

label_cols = [c for c in bhrrc_by_factory.columns if c.startswith("label_")]
print("\nLabel distribution (number of factories with at least one allegation):")
print(bhrrc_by_factory[label_cols].sum().rename(lambda x: x.replace("label_", "")))

# ----------------------------------------------------------------
# 8. Fuzzy match BHRRC factories to OSH registry
#
# Factory names are often written slightly differently across
# datasets. Fuzzy string matching (Levenshtein distance) finds
# the closest OSH name for each BHRRC factory.
# Threshold of 90/100 keeps only confident matches.
# ----------------------------------------------------------------

FUZZY_THRESHOLD = 90
osh_names = osh["name_lower"].tolist()

def match_to_osh(factory_name_lower):
    result = process.extractOne(factory_name_lower, osh_names, score_cutoff=FUZZY_THRESHOLD)
    if result:
        best_name, score = result
        idx = osh_names.index(best_name)
        return osh.iloc[idx]["os_id"], osh.iloc[idx]["name"], score
    return None, None, None

bhrrc_by_factory["factory_lower"] = bhrrc_by_factory["factory_name"].str.lower().str.strip()

bhrrc_by_factory[["os_id", "osh_name_matched", "match_score"]] = bhrrc_by_factory["factory_lower"].apply(
    lambda name: pd.Series(match_to_osh(name))
)

matched   = bhrrc_by_factory["os_id"].notna().sum()
unmatched = bhrrc_by_factory["os_id"].isna().sum()

print(f"\nOSH matching results: {matched} matched, {unmatched} unmatched")
print("\nSample matches:")
print(
    bhrrc_by_factory[bhrrc_by_factory["os_id"].notna()][
        ["factory_name", "osh_name_matched", "match_score"]
    ].head(8).to_string(index=False)
)

# ----------------------------------------------------------------
# 9. Save outputs for Stage 2
# ----------------------------------------------------------------

news.to_csv("stage1_news.csv", index=False, encoding="utf-8-sig")
osh.to_csv("stage1_osh.csv", index=False, encoding="utf-8-sig")
bhrrc_by_factory.to_csv("stage1_bhrrc_labels.csv", index=False, encoding="utf-8-sig")
bhrrc.to_csv("stage1_bhrrc_full.csv", index=False, encoding="utf-8-sig")

print("\nStage 1 done. Four files saved and ready for Stage 2.")

Re-scraping article content...
[1/1441] Workers at Chai Moon Sports (Myanmar) Forced to Sign Warning
[2/1441] Foreign Female Manager at Jiangsu Soho Myanmar Accused of Su
[3/1441] Earnest Myanmar Wool Factory Forces Daily Overtime and Press
[4/1441] Dishang Fashion (Myanmar) Garment Factory Forcing Workers to
[5/1441] Dagon Talent Garment Forces Workers to Do 4 Hours Overtime W
[6/1441] Factory Manager Physically Assaults Workers at Kyay Oh Gyi E
[7/1441] TMA Garment Factory Supervisor Deducting Piece-Rate Wages
[8/1441] Zhong Ying Garment Factory Forces Workers to Work Overtime, 
[9/1441] Lat War Co., Ltd (3) to Close, Says Compensation Will Only B
[10/1441] Handa 2 Factory Dispute – 14,000 Kyat Garment Fee Deduction 
[11/1441] Male Line Supervisor at Justdo Myanmar Garment Factory Accus
[12/1441] 21 Leaders Fired After Workers Stage Demands at Handa 2 Fact
[13/1441] Hope One Co., Ltd Garment Factory Requires Overtime Beyond A
[14/1441] Nay Shwe Win Factory Dismisses Labour Union Lead